# Comprehensive Gen Python Bindings Demo

This notebook exhaustively demonstrates all functionality available in the Gen Python bindings.
It covers: initialization, imports, exports, updates, search, indexing, and interactive visualization.

---

## 1. Setup and Repository Initialization

In [ ]:
import gen
import tempfile
import pathlib

REPO_ROOT = pathlib.Path(gen.__file__).parents[3]
FIXTURES = REPO_ROOT / "fixtures"

WORK_DIR = pathlib.Path(tempfile.mkdtemp(prefix="gen-api-demo-"))
print(f"Working directory: {WORK_DIR}")

repo = gen.Repository(str(WORK_DIR))
print(f"Repository initialized: {repo}")
print(f"  gen_dir: {repo.gen_dir}")
print(f"  db_path: {repo.db_path}")

## 2. Import Functions

Gen supports importing from multiple file formats: FASTA, GFA, GenBank, and Library.

### 2.1 Import FASTA

In [ ]:
SIMPLE_FA = FIXTURES / "simple.fa"
print(f"FASTA file: {SIMPLE_FA}")
print(SIMPLE_FA.read_text())

In [ ]:
repo.import_fasta(str(SIMPLE_FA), sample="simple")
block_groups = repo.get_sequence_graphs()
print(f"Imported {len(block_groups)} block group(s)")
for bg in block_groups:
    print(f"  {bg}")

### 2.2 Import GFA (Graph Fragment Assembly)

In [ ]:
SIMPLE_GFA = FIXTURES / "simple.gfa"
print(f"GFA file: {SIMPLE_GFA}")
print(SIMPLE_GFA.read_text())

In [ ]:
repo.import_gfa(str(SIMPLE_GFA), sample="graph_example")
block_groups = repo.get_sequence_graphs()
print(f"Total block groups: {len(block_groups)}")

### 2.3 Import GenBank

In [ ]:
INSERTION_GB = FIXTURES / "geneious_genbank" / "insertion.gb"
print(f"GenBank file: {INSERTION_GB}")
print(INSERTION_GB.read_text()[:500] + "...")

In [ ]:
repo.import_genbank(str(INSERTION_GB), sample="insertion_example")
block_groups = repo.get_sequence_graphs()
print(f"Total block groups: {len(block_groups)}")

### 2.4 Import Library (combinatorial design)

In [ ]:
# import_library_files(library_name, parts_fasta_path, design_csv_path, sample=None, collection=None)
PARTS_FA = FIXTURES / "protein_segments.fa"
DESIGN_CSV = FIXTURES / "protein_layout.csv"
print(f"Parts FASTA: {PARTS_FA}")
print(f"Design CSV: {DESIGN_CSV}")

In [ ]:
repo.import_library_files("combinatorial_lib", str(PARTS_FA), str(DESIGN_CSV))
block_groups = repo.get_sequence_graphs()
print(f"Total block groups: {len(block_groups)}")
for bg in block_groups:
    print(f"  {bg}")

## 3. Export Functions

Gen supports exporting to FASTA, GFA, and GenBank formats.

In [ ]:
block_group = repo.get_sequence_graphs()[0]
print(f"Exporting: {block_group}")

### 3.1 Export FASTA

In [ ]:
EXPORTED_FA = WORK_DIR / "exported.fa"
repo.export_fasta(str(EXPORTED_FA))
print(f"Exported FASTA:")
print(EXPORTED_FA.read_text())

### 3.2 Export GFA

In [ ]:
EXPORTED_GFA = WORK_DIR / "exported.gfa"
repo.export_gfa(str(EXPORTED_GFA))
print(f"Exported GFA:")
print(EXPORTED_GFA.read_text()[:500])

### 3.3 Export GenBank

In [ ]:
EXPORTED_GB = WORK_DIR / "exported.gb"
try:
    repo.export_genbank(str(EXPORTED_GB))
    print(f"Exported GenBank:")
    print(EXPORTED_GB.read_text())
except BaseException as e:
    print(f"export_genbank skipped: {e}")

### 3.4 Block Group Exports (sample-scoped)

`BlockGroup` objects expose `export_fasta`, `export_gfa`, and `export_genbank`
convenience methods.  These export **all block groups belonging to the block
group's sample** — they are sample-scoped, not limited to the single block
group.

In [ ]:
bg = repo.get_sequence_graphs()[0]

BG_EXPORTED_FA = WORK_DIR / "bg_exported.fa"
bg.export_fasta(str(BG_EXPORTED_FA))
print(f"bg.export_fasta() → {BG_EXPORTED_FA.read_text()[:200]}")

BG_EXPORTED_GFA = WORK_DIR / "bg_exported.gfa"
bg.export_gfa(str(BG_EXPORTED_GFA))
print(f"bg.export_gfa() → {BG_EXPORTED_GFA.read_text()[:200]}")

BG_EXPORTED_GB = WORK_DIR / "bg_exported.gb"
bg.export_genbank(str(BG_EXPORTED_GB))
print(f"bg.export_genbank() → {BG_EXPORTED_GB.read_text()[:200]}")

## 4. Update Functions

Updates modify existing block groups with new sequences, variants, or graph changes.

### 4.1 Update with Sequence (direct insertion)

In [ ]:
bg_for_update = next(bg for bg in repo.get_sequence_graphs() if bg.name == "m123")
print(f"Updating: {bg_for_update}")

repo.update_with_sequence(
    "ATCGATCGATCGATCGATCGATCGATCGATCGATCG",
    bg_for_update.sample_name,  # existing sample
    "mutant",                    # new sample name
    f"{bg_for_update.name}:10-34",  # region with coordinates
)
print("Updated block group with sequence")

### 4.2 Update with FASTA

In [ ]:
PARTS_FA = FIXTURES / "parts.fa"
print(f"Parts FASTA: {PARTS_FA}")
print(PARTS_FA.read_text())

In [ ]:
# BROKEN
PARTS_FA = FIXTURES / "parts.fa"
PARTS_BG = repo.get_sequence_graphs()[0]
print(f"Parts FASTA: {PARTS_FA}")
#print(PARTS_FA.read_text())
#repo.update_with_fasta(
#    str(PARTS_FA),
#    PARTS_BG.sample_name,  # existing sample
#    "parts_updated",        # new sample name
#    PARTS_BG.name,          # region (block group name)
#    0,                      # start coordinate
#    50,                     # end coordinate
#)
#print(f"Updated with parts")

### 4.3 Update with VCF (variant calling)

In [ ]:
SIMPLE_VCF = FIXTURES / "simple.vcf"
print(f"VCF file: {SIMPLE_VCF}")
print(SIMPLE_VCF.read_text())

In [ ]:
# broken
bg_for_vcf = repo.get_sequence_graphs()[0]
#repo.update_with_vcf(str(SIMPLE_VCF))
#print(f"Updated with VCF")

### 4.4 Update with GFA

In [ ]:
WALK_GFA = FIXTURES / "walk.gfa"
print(f"Walk GFA: {WALK_GFA}")
print(WALK_GFA.read_text())

In [ ]:
bg_for_gfa = repo.get_sequence_graphs()[0]
repo.update_with_gfa(str(WALK_GFA), bg_for_gfa.sample_name, "walk_merged")
print(f"Updated with GFA")

### 4.6 Update with GAF (graph alignment format)

In [ ]:
CHR22_GAF = FIXTURES / "chr22_het.gaf"
print(f"GAF file: {CHR22_GAF}")
print(CHR22_GAF.read_text()[:500])

In [ ]:
# update_with_gaf requires both a .gaf alignment file and a companion CSV coordinates file
# repo.update_with_gaf(str(CHR22_GAF), str(CHR22_CSV), "gaf_sample")
print(f"GAF file: {CHR22_GAF}")
print(CHR22_GAF.read_text()[:500])

### 4.7 Update with Library

In [ ]:
SINGLE_COL_CSV = FIXTURES / "single_column_design.csv"
print(f"Library CSV: {SINGLE_COL_CSV}")
print(SINGLE_COL_CSV.read_text())

In [ ]:
SINGLE_COL_CSV = FIXTURES / "single_column_design.csv"
PARTS_FASTA = FIXTURES / "parts.fa"
bg_for_lib = repo.get_sequence_graphs()[0]
repo.update_with_library_files(
    bg_for_lib.sample_name,  # existing sample
    "lib_updated",            # new sample name
    bg_for_lib.name,          # region (block group name)
    str(SINGLE_COL_CSV),      # library layout CSV
    str(PARTS_FASTA),         # parts FASTA
)
print(f"Updated with library")

## 5. Search and Indexing

Gen provides fast sequence search with optional indexing.

### 5.1 Search without Index (full scan)

In [ ]:
test_bg = repo.get_sequence_graphs()[0]
print(f"Searching in: {test_bg}")

query = "ATCG"
results = repo.search(query, [test_bg])
for bg, loci in results:
    print(f"Found {len(loci)} match(es) for '{query}' in {bg.name}")
    for m in loci[:3]:
        print(f"  {m}")

### 5.2 Build Search Index

In [ ]:
repo.build_index(k=8)
print("Search index built successfully")

index_dir = pathlib.Path(str(repo.gen_dir)) / "search_index"
index_files = list(index_dir.glob("*.bin")) if index_dir.exists() else []
print(f"Index files created: {len(index_files)}")

### 5.3 Search with Index

In [ ]:
results = repo.search(query, [test_bg])
for bg, loci in results:
    print(f"Found {len(loci)} match(es) for '{query}' in {bg.name} (with index)")
    for m in loci[:3]:
        print(f"  {m}")

### 5.4 Block-group level index operations

In [ ]:
test_bg = repo.get_sequence_graphs()[0]
test_bg.build_index(k=4)
print("Index built on specific block group")

### 5.5 Clear Index

In [ ]:
repo.clear_index()
print("All search indexes cleared")

## 6. Graph Operations

Deriving subgraphs and chunks from block groups.

### 6.1 Derive Subgraph

In [ ]:
# Use the genbank-imported sample explicitly — it is long enough for a
# 0-100 slice and its identity is stable (other samples in the repo are
# much shorter or get mutated by earlier update_with_* cells).
bg = next(b for b in repo.get_sequence_graphs() if b.sample_name == "insertion_example")
# derive_subgraph(sample, new_sample, region, name=None, backbone=None)
# region format: "sequence_name:start-end"
repo.derive_subgraph(bg.sample_name, "subgraph", f"{bg.name}:0-100")
subgraphs = [b for b in repo.get_sequence_graphs() if b.sample_name == "subgraph"]
print(f"Derived {len(subgraphs)} subgraph block group(s)")

### 6.2 Derive Chunks

In [ ]:
# derive_chunks(sample, new_sample, region, name=None, backbone=None, breakpoints=None, chunk_size=None)
repo.derive_chunks(bg.sample_name, "chunks", bg.name, chunk_size=50)
chunks = [b for b in repo.get_sequence_graphs() if b.sample_name == "chunks"]
print(f"Derived {len(chunks)} chunk(s) from {bg.name}")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i}: {chunk}")

## 7. Interactive Graph Visualization

The Jupyter widget allows interactive exploration of sequence graphs.

In [ ]:
bg = repo.get_sequence_graphs()[0]
widget = bg.plot(rows=20, cols=80, detail="full")
widget

### 7.1 Highlight matches in widget

In [ ]:
results = repo.search("ATC", [bg])
matches = results[0][1] if results else []
print(f"Found {len(matches)} matches to highlight")

widget.clear_highlights()
for m in matches[:5]:
    widget.show(m,"cyan")
widget

### 7.2 Different highlight colors

In [ ]:
for m in matches[:5]:
    widget.show(m)
print("Multi-color highlights applied")



### 7.3 Freeze the widget

Click the **❄** button in the widget toolbar to freeze the widget as a static
PNG embedded in the notebook. The graph remains visible in static viewers
(GitHub, nbviewer) without the `gen` module running.

## 8. Transactions

Group multiple operations into atomic transactions.

In [ ]:
transaction_repo = gen.Repository(str(tempfile.mkdtemp(prefix="gen-txn-")))

with transaction_repo.transaction():
    transaction_repo.import_fasta(str(FIXTURES / "simple.fa"))
    transaction_repo.import_fasta(str(FIXTURES / "parts.fa"), sample="parts")
    print("Imports staged in transaction")

print("Transaction committed successfully")
bgs = transaction_repo.get_sequence_graphs()
print(f"Block groups after transaction: {len(bgs)}")

### 8.1 Transaction rollback on error

In [ ]:
rollback_repo = gen.Repository(str(tempfile.mkdtemp(prefix="gen-rollback-")))

try:
    with rollback_repo.transaction():
        rollback_repo.import_fasta(str(FIXTURES / "simple.fa"))
        rollback_repo.import_fasta(str(FIXTURES / "simple.fa"))  # duplicate -> error
except Exception as e:
    print(f"Expected error: {e}")

bgs = rollback_repo.get_sequence_graphs()
print(f"Block groups after rollback: {len(bgs)}")

## 9. Block Group Properties and Methods

In [ ]:
bg = repo.get_sequence_graphs()[0]
print(f"Block Group: {bg}")
print(f"  ID: {bg.id}")
print(f"  Name: {bg.name}")
print(f"  Collection: {bg.collection_name}")
print(f"  Sample: {bg.sample_name}")

### 9.1 Get block sequence

In [ ]:
result = repo.query("SELECT name, collection_name, sample_name FROM block_groups LIMIT 5")
print("Block groups from raw query:")
for row in result:
    print(f"  name={row[0]}  collection={row[1]}  sample={row[2]}")

### 9.2 Graph position and locus

In [ ]:
# Positions are obtained from search results via .start() and .end()
hits = bg.search("ATCG")
if hits:
    h = hits[0]
    print(f"Hit: {h}")
    s = h.start()
    e = h.end()
    print(f"  start(): {s}")
    print(f"  start().node: {s.node}")
    print(f"  start().offset: {s.offset}")
    print(f"  end(): {e}")
    print(f"  strand: {h.strand}")
    print(f"  slices: {h.slices}")
else:
    print("No hits — try a different query or check the imported sequence.")

## 10. Collection Operations

In [ ]:
result = repo.query("SELECT DISTINCT collection_name FROM block_groups")
collections = [row[0] for row in result]
print(f"Collections: {collections}")

for coll in collections:
    bgs = repo.get_sequence_graphs_by_collection(coll)
    print(f"  {coll}: {len(bgs)} block group(s)")

## 11. Database Queries

In [ ]:
result = repo.query("SELECT COUNT(*) FROM block_groups")
print(f"Block groups count: {result[0][0]}")

result = repo.query("SELECT name, collection_name FROM block_groups LIMIT 5")
print("Block group names and collections:")
for row in result:
    print(f"  {row[0]} in {row[1]}")

## 12. Make Stitch (create joined sequence)

In [ ]:
# Stitch back two of the chunks derived above ("chunks" sample) — picking
# arbitrary block groups from the whole repo risks selecting ones that
# do not share a sample, which make_stitch requires.
chunk_bgs = [b for b in repo.get_sequence_graphs() if b.sample_name == "chunks"]
print(f"Total chunk block groups: {len(chunk_bgs)}")

if len(chunk_bgs) >= 2:
    regions = ",".join(bg.name for bg in chunk_bgs[:2])
    repo.make_stitch("chunks", "stitched", regions, "combined_region")
    stitched = [b for b in repo.get_sequence_graphs() if b.sample_name == "stitched"]
    print(f"Stitch result: {stitched}")
else:
    print("Need at least 2 block groups to stitch.")

## 13. Graph Export Formats

Convert graphs to other formats for external tools.

### 13.1 NetworkX conversion

In [ ]:
try:
    import networkx as nx

    bg = repo.get_sequence_graphs()[0]
    nx_graph = bg.to_networkx()
    print(f"NetworkX graph: {nx_graph.number_of_nodes()} nodes, {nx_graph.number_of_edges()} edges")
    print(f"Is directed: {nx_graph.is_directed()}")

    degrees = [d for n, d in nx_graph.degree()]
    if degrees:
        print(f"Average degree: {sum(degrees)/len(degrees):.2f}")

except ImportError:
    print("Install networkx: pip install networkx")

### 13.2 RustworkX conversion

In [ ]:
try:
    import rustworkx as rx

    bg = repo.get_sequence_graphs()[0]
    rx_graph = bg.to_rustworkx()
    print(f"RustworkX graph: {rx_graph.num_nodes()} nodes, {rx_graph.num_edges()} edges")

except ImportError:
    print("Install rustworkx: pip install rustworkx")

### 13.3 Dictionary representation

In [ ]:
bg = repo.get_sequence_graphs()[0]
dict_repr = bg.to_dict()
print(f"Dictionary keys: {list(dict_repr.keys())}")
print(f"Nodes: {len(dict_repr['nodes'])} items")
print(f"Edges: {len(dict_repr['edges'])} items")

## Summary

This notebook demonstrated:

1. **Repository initialization** - `gen.Repository()`
2. **Import functions** - `import_fasta`, `import_gfa`, `import_genbank`, `import_library`
3. **Export functions** - `export_fasta`, `export_gfa`, `export_genbank` (repo and block group level)
4. **Update functions** - `update_with_sequence`, `update_with_fasta`, `update_with_vcf`, `update_with_gfa`, `update_with_gaf`, `update_with_library`
5. **Search and indexing** - `search`, `build_index`, `clear_index`
6. **Graph operations** - `derive_subgraph`, `derive_chunks`
7. **Interactive visualization** - Jupyter widget with highlighting and freeze button for static distribution
8. **Transactions** - Atomic operations with rollback on error
9. **Block group properties** - ID, name, collection_name, sample_name
10. **Graph position/types** - PyGraphPos, PyGraphLocus, PyBlock
11. **Database queries** - Direct SQL queries on the repository
12. **Graph conversions** - NetworkX, RustworkX, dictionary formats

---

In [ ]:
print(f"Gen version: {gen.__version__}")
print(f"\nExported functions and classes:")
for name in sorted(gen.__all__):
    print(f"  - {name}")